In [ ]:
!pip install sentencepiece

In [ ]:
%%writefile config.py

import torch


def get_hyperparams():
    # --- Settings you can change ---
    batch_size = 8  # Physical batch size (fits in 16GB VRAM with 1024 block size)
    max_iters = 10000  # We don't have enough compute time on Kaggle for this before you reach the 12 hour limit, ~5000 is achievable in one go
    learning_rate = 3e-4

    # Target effective batch size: 128 sequences
    target_batch_size = 128
    grad_accum = target_batch_size // batch_size

    CONFIG = {
        "batch_size": batch_size,
        "block_size": 1024,
        "max_iters": max_iters,
        "eval_interval": 20,
        "learning_rate": learning_rate,
        "device": 'cuda' if torch.cuda.is_available() else 'cpu', # Train using GPU if available, else CPU (or you could change to 'mps' for Mac)
        "eval_iters": 20,
        "n_embd": 768,
        "n_head": 12,
        "n_layer": 12,
        "dropout": 0.05,
        "gradient_accumulation_steps": grad_accum,
        "lr_decay_iters": max_iters,
        "min_lr": learning_rate / 10,
        "warmup_iters": 500
    }
    return CONFIG

In [ ]:
%%writefile multi_head_attention.py

import torch
import torch.nn as nn
from torch.nn import functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, dropout, block_size):
        super().__init__()
        self.n_head = num_heads
        self.head_size = head_size
        self.n_embd = n_embd
        
        # COMBINED PROJECTION: 
        # Instead of 3 small Linear layers per head, we do 1 giant Linear layer 
        # that calculates Query, Key, and Value for ALL heads at the same time.
        # Output size is 3 * n_embd because we need Q, K, and V.
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        
        # Output projection
        self.c_proj = nn.Linear(n_embd, n_embd)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size() # Batch, Time, Channels (n_embd)

        # 1. Calculate Q, K, V for all heads in one go
        # shape: (B, T, n_embd) -> (B, T, 3 * n_embd)
        qkv = self.c_attn(x)
        
        # 2. Split the result into Query, Key, Value
        q, k, v = qkv.split(self.n_embd, dim=2)
        
        # 3. Reshape for Multi-Head Attention
        # Transform from (B, T, C) -> (B, n_head, T, head_size)
        # This allows us to treat "Heads" as a batch dimension for parallel processing
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)

        # 4. Flash Attention (Optimized Kernel)
        # This is the single fastest way to do attention in PyTorch
        y = F.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=None, 
            dropout_p=self.attn_dropout.p if self.training else 0, 
            is_causal=True
        )
        
        # 5. Re-assemble heads
        # (B, n_head, T, head_size) -> (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        # 6. Final projection
        y = self.resid_dropout(self.c_proj(y))
        
        return y

In [ ]:
%%writefile feed_forward.py

import torch.nn as nn

class FeedForward(nn.Module):
    # this layer figures out what the relationships between words are
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
%%writefile block.py

import torch.nn as nn

from feed_forward import FeedForward
from multi_head_attention import MultiHeadAttention

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    # this is just one round of running the attention heads and then running our feedforward

    def __init__(self, n_embd, n_head, dropout, block_size):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(num_heads=n_head, head_size=head_size, n_embd=n_embd, dropout=dropout, block_size=block_size)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [ ]:
%%writefile gpt_language_model.py

import torch
import torch.nn as nn
from torch.nn import functional as F
from block import Block


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_head, n_layer, device, dropout):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head, dropout=dropout, block_size=block_size) for _ in range(n_layer)]) # the number of blocks is the amount of times we will gather info and figure out the relationships
        self.ln_f = nn.LayerNorm(n_embd)  # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.device = device
        self.block_size = block_size

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx)  # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=self.device))  # (T,C)
        x = tok_emb + pos_emb  # (B,T,C)
        x = self.blocks(x)  # (B,T,C)
        x = self.ln_f(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.7):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -self.block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :]  # becomes (B, C)

            # Apply temperature (Scales the probabilities)
            # Higher temp (1.0) = More random/creative
            # Lower temp (0.5) = More predictable/logical
            logits = logits / temperature

            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx

In [ ]:
import math
import torch
import sentencepiece as spm
import os
import time
from gpt_language_model import GPTLanguageModel
from config import get_hyperparams
import numpy as np

# --- 1. SETUP PATHS ---
# Verify this matches your Kaggle dataset name exactly!
DATASET_PATH = "/kaggle/input/datasets/agmadaasa/llm-data"

train_path = os.path.join(DATASET_PATH, "train.bin")
val_path = os.path.join(DATASET_PATH, "val.bin")
tokenizer_path = os.path.join(DATASET_PATH, "token.model")

# Standard output path (writable)
model_save_path = "model_weights.pth"
model_info_path = "model_info.txt"

CONFIG = get_hyperparams()

# --- 2. HYPERPARAMETERS ---
batch_size = CONFIG["batch_size"]
block_size = CONFIG["block_size"]
learning_rate = CONFIG["learning_rate"]
dropout = CONFIG["dropout"]
n_embd = CONFIG["n_embd"]
n_head = CONFIG["n_head"]
n_layer = CONFIG["n_layer"]
max_iters = CONFIG["max_iters"]
eval_interval = CONFIG["eval_interval"]
eval_iters = CONFIG["eval_iters"]
device = CONFIG["device"]
gradient_accumulation_steps = CONFIG["gradient_accumulation_steps"]
warmup_iters = CONFIG["warmup_iters"]
lr_decay_iters = CONFIG["lr_decay_iters"]
min_lr = CONFIG["min_lr"]

print(f"Using device: {device}")


# --- 3. DATA LOADING ---
def get_data_memmap(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {path}. Check your Kaggle dataset path.")
    # mode='r' means read-only
    return np.memmap(path, dtype=np.uint16, mode='r')


print("Memory-mapping binary datasets...")
try:
    train_data = get_data_memmap(train_path)
    val_data = get_data_memmap(val_path)
except FileNotFoundError as e:
    print(f"\nCRITICAL ERROR: {e}")
    print(f"Listed files in {DATASET_PATH}: {os.listdir(DATASET_PATH)}")
    exit()

print(f"Train size: {len(train_data):,} tokens")
print(f"Val size: {len(val_data):,} tokens")

# load tokenizer
sp = spm.SentencePieceProcessor()
sp.load(tokenizer_path)
vocab_size = sp.get_piece_size()


def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i + block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i + 1:i + block_size + 1]).astype(np.int64)) for i in ix])
    # pinning memory is a slight optimization for CPU->GPU transfer
    if device == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            # Mixed precision context isn't strictly necessary for eval but good practice
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)


# --- 4. MODEL INITIALIZATION ---
print("Initializing model...")
model = GPTLanguageModel(vocab_size=vocab_size, n_embd=n_embd, n_layer=n_layer, n_head=n_head, block_size=block_size,
                         device=device, dropout=dropout)
model = model.to(device)

# OPTIMIZATION: Compile the model (Requires PyTorch 2.0+, standard on Kaggle)
print("Compiling model (this may take a minute at the start)...")
model = torch.compile(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# OPTIMIZATION: Initialize GradScaler for Mixed Precision Training
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

# --- 5. TRAINING LOOP ---
print(f"Starting training on {device}...")
print(f"Effective batch size: {batch_size * gradient_accumulation_steps}")
start_time = time.time()
final_losses = ""

for iter in range(max_iters):
    t0 = time.time()

    # Update Learning Rate
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Gradient Accumulation
    for micro_step in range(gradient_accumulation_steps):
        xb, yb = get_batch('train')

        # OPTIMIZATION: Mixed Precision Forward Pass
        with torch.cuda.amp.autocast(dtype=torch.float16, enabled=(device == 'cuda')):
            logits, loss = model(xb, yb)
            loss = loss / gradient_accumulation_steps  # scale loss

        # OPTIMIZATION: Mixed Precision Backward Pass
        scaler.scale(loss).backward()

    # OPTIMIZATION: Stepping with Scaler
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    t1 = time.time()
    dt = t1 - t0

    # Logging
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(
            f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.2e}, time {dt:.4f}s")

    if iter == max_iters - 1:
        final_losses = f"Final Train Loss: {losses['train']:.4f}, Final Val Loss: {losses['val']:.4f}, Final Learning Rate: {lr:.2e}"

    # Save model every 500 steps
    if iter > 0 and iter % 500 == 0:
        print(f"--> Saving rolling checkpoint...")
        torch.save(model.state_dict(), "latest_checkpoint.pth")

end_time = time.time()
print(f"Training finished in {end_time - start_time:.2f} seconds.")

torch.save(model.state_dict(), model_save_path)
print("Model saved successfully!")

# --- 6. INFERENCE ---
print("\nGenerating sample output:")
start_prompt = "<|user|>What is 2 + 2?<|end|>\n<|assistant|>"
context = torch.tensor([sp.encode_as_ids(start_prompt)], dtype=torch.long, device=device)
generated_ids = model.generate(context, max_new_tokens=200)[0].tolist()
sample_output = sp.decode(generated_ids)
print(sample_output)

model_size = sum(p.numel() for p in model.parameters()) / 1e6

with open(model_info_path, "w", encoding="utf-8") as f:
    f.write(
        f"Model Size:{model_size:.3f}M parameters\nTraining finished in {end_time - start_time:.2f} seconds\n{final_losses}\nHyperparamter Info: {CONFIG}\nSample Output:\n{sample_output}")